# JobHop v2 LLM Filtering Pilot

This Colab notebook is a small preliminary experiment for an LLM filtering stage.

Protocol in this notebook:

1. Load JobHop from Hugging Face: `aida-ugent/JobHop`.
2. Load ESCO metadata from Hugging Face: `danieldux/ESCO`.
3. Build last-step next-occupation validation cases.
4. Use BGE-M3 to retrieve top-K candidate ESCO occupations.
5. Ask an LLM to score top candidates as plausible next occupations.
6. Filter candidates by a validation-only threshold and compare retrieval metrics before/after filtering.

This is a pilot notebook only. It does not modify the cluster scripts or the main experimental protocol.

In [ ]:
!pip -q install datasets pandas pyarrow requests tqdm sentence-transformers openai

## Configuration

For an OpenAI-compatible LLM endpoint, set:

- `OPENAI_API_KEY`
- optionally `OPENAI_BASE_URL`
- `LLM_MODEL`

If `RUN_LLM = False`, the notebook only runs the dense retrieval baseline.

In [ ]:
import os

DATASET_ID = "aida-ugent/JobHop"
ESCO_DATASET_ID = "danieldux/ESCO"

MAX_EVAL_CASES = 50
RANDOM_SEED = 7

EMBEDDING_MODEL = "BAAI/bge-m3"
EMBED_BATCH_SIZE = 32
RETRIEVE_K = 50

# Number of dense top candidates sent to the LLM per query.
LLM_TOP_K = 30

# Keep candidates whose LLM score is at least this value.
# 0 = not plausible, 1 = weakly plausible, 2 = plausible, 3 = highly plausible.
FILTER_THRESHOLD = 1

RUN_LLM = True
LLM_MODEL = os.environ.get("LLM_MODEL", "gpt-4o-mini")
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL")

# Optional for China/HF mirror environments, e.g. os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

METRIC_KS = [1, 5, 10]

print({
    "dataset": DATASET_ID,
    "esco_metadata": ESCO_DATASET_ID,
    "max_eval_cases": MAX_EVAL_CASES,
    "embedding_model": EMBEDDING_MODEL,
    "retrieve_k": RETRIEVE_K,
    "llm_top_k": LLM_TOP_K,
    "filter_threshold": FILTER_THRESHOLD,
    "run_llm": RUN_LLM,
    "llm_model": LLM_MODEL,
})

In [ ]:
import json
import random
import re
import time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import requests
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "cuda", torch.version.cuda, "available", torch.cuda.is_available(), "device", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))

## Data Loading and JobHop v2 Preprocessing

In [ ]:
SPLITS = ("train", "validation", "test")
REMOVE_UNKNOWN = True
COLLAPSE_CONSECUTIVE_DUPLICATES = True

def hf_tree(path=""):
    api = f"https://huggingface.co/api/datasets/{DATASET_ID}/tree/main"
    url = api if not path else f"{api}/{path}"
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    return response.json()

def list_parquet_files():
    files = []
    stack = [""]
    while stack:
        path = stack.pop()
        for item in hf_tree(path):
            item_path = item.get("path") or item.get("name")
            item_type = item.get("type")
            if not item_path:
                continue
            if item_type == "directory":
                stack.append(item_path)
            elif item_path.endswith(".parquet"):
                files.append(item_path)
    return sorted(files)

def split_from_path(path):
    lower = path.lower()
    for split in SPLITS:
        if re.search(rf"(^|/|_|-)({split})(/|_|-|\.|$)", lower):
            return split
    if "val" in lower:
        return "validation"
    return None

def hf_resolve_url(path):
    return f"https://huggingface.co/datasets/{DATASET_ID}/resolve/main/{path}"

def load_split(split, split_files):
    paths = split_files.get(split, [])
    if not paths:
        raise RuntimeError(f"No parquet files found for split={split}.")
    frames = [pd.read_parquet(hf_resolve_url(path)) for path in paths]
    df = pd.concat(frames, ignore_index=True)
    print(split, df.shape, paths)
    return df

def standardize_jobhop_schema(df):
    df = df.copy()
    rename = {}
    if "resume_id" in df.columns and "person_id" not in df.columns:
        rename["resume_id"] = "person_id"
    if "university_level" in df.columns and "university_studies" not in df.columns:
        rename["university_level"] = "university_studies"
    if rename:
        df = df.rename(columns=rename)
    required = ["person_id", "matched_code"]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise RuntimeError(f"Missing required columns: {missing}; available columns={list(df.columns)}")
    return df

def normalize_code(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    if not text or text.lower() == "unknown":
        return None
    return text

def parse_year_quarter(value):
    if pd.isna(value):
        return (9999, 9)
    text = str(value)
    year_match = re.search(r"(19|20)\d{2}", text)
    year = int(year_match.group(0)) if year_match else 9999
    quarter_match = re.search(r"[Qq]([1-4])", text)
    quarter = int(quarter_match.group(1)) if quarter_match else 9
    return (year, quarter)

def make_trajectories(df):
    work = df.copy()
    work["matched_code"] = work["matched_code"].map(normalize_code)
    if REMOVE_UNKNOWN:
        work = work.dropna(subset=["matched_code"])
    work["_start_sort"] = work["start_date"].map(parse_year_quarter) if "start_date" in work.columns else [(9999, 9)] * len(work)
    work["_end_sort"] = work["end_date"].map(parse_year_quarter) if "end_date" in work.columns else [(9999, 9)] * len(work)
    sort_cols = ["person_id", "_start_sort", "_end_sort", "matched_code"]
    work = work.sort_values(sort_cols, kind="mergesort")
    trajectories = {}
    for person_id, group in work.groupby("person_id", sort=False):
        seq = []
        for _, row in group.iterrows():
            code = row["matched_code"]
            item = {
                "code": code,
                "university_studies": row.get("university_studies", None),
            }
            if COLLAPSE_CONSECUTIVE_DUPLICATES and seq and seq[-1]["code"] == code:
                continue
            seq.append(item)
        if len(seq) >= 2:
            trajectories[person_id] = seq
    return trajectories

def make_last_step_cases(trajectories):
    cases = []
    for person_id, seq in trajectories.items():
        cases.append({
            "person_id": person_id,
            "history": seq[:-1],
            "target": seq[-1]["code"],
            "university_studies": seq[-1].get("university_studies"),
        })
    return cases

In [ ]:
def normalize_esco_code(value):
    if pd.isna(value):
        return None
    return str(value).strip()

def load_esco_metadata():
    frames = []
    urls = [
        "https://huggingface.co/datasets/danieldux/ESCO/resolve/main/data/en/train.parquet",
        "https://huggingface.co/datasets/danieldux/ESCO/resolve/main/data/en/test.parquet",
    ]
    for url in urls:
        try:
            frames.append(pd.read_parquet(url))
        except Exception as exc:
            print("direct parquet load failed", url, exc)
    if not frames:
        ds = load_dataset("danieldux/ESCO", split="train")
        frames.append(ds.to_pandas())
    df = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["ESCO_CODE"])
    lookup = {}
    for _, row in df.iterrows():
        code = normalize_esco_code(row.get("ESCO_CODE"))
        if code:
            lookup[code] = {
                "label": str(row.get("ESCO_OCCUPATION", "") or ""),
                "description": str(row.get("ESCO_DESCRIPTION", "") or ""),
            }
    return lookup

def occupation_label(metadata_lookup, code):
    meta = metadata_lookup.get(code, {})
    return meta.get("label") or f"ESCO code {code}"

def candidate_text(metadata_lookup, code):
    meta = metadata_lookup.get(code, {})
    label = meta.get("label") or f"ESCO code {code}"
    description = meta.get("description") or ""
    pieces = [f"ESCO occupation code: {code}.", f"Occupation title: {label}."]
    if description:
        pieces.append(f"Occupation description: {description}")
    return " ".join(pieces)

def query_text(metadata_lookup, case):
    history = "; ".join(occupation_label(metadata_lookup, item["code"]) for item in case["history"])
    codes = "; ".join(item["code"] for item in case["history"])
    edu = case.get("university_studies")
    edu_text = ""
    if edu is not None and not pd.isna(edu):
        edu_text = f" University studies: {edu}."
    return f"Career history occupations: {history}. Career history ESCO codes: {codes}.{edu_text} Predict the next ESCO occupation."

In [ ]:
parquet_files = list_parquet_files()
split_files = {split: [] for split in SPLITS}
for path in parquet_files:
    split = split_from_path(path)
    if split in split_files:
        split_files[split].append(path)

raw = {split: standardize_jobhop_schema(load_split(split, split_files)) for split in SPLITS}
metadata_lookup = load_esco_metadata()
trajectories = {split: make_trajectories(raw[split]) for split in SPLITS}

candidate_codes = sorted({item["code"] for seq in trajectories["train"].values() for item in seq})
candidate_set = set(candidate_codes)
validation_cases = make_last_step_cases(trajectories["validation"])
validation_cases = [case for case in validation_cases if case["target"] in candidate_set]

rng = random.Random(RANDOM_SEED)
validation_cases = rng.sample(validation_cases, min(MAX_EVAL_CASES, len(validation_cases)))

candidate_texts = [candidate_text(metadata_lookup, code) for code in candidate_codes]
validation_query_texts = [query_text(metadata_lookup, case) for case in validation_cases]

print("candidate occupations:", len(candidate_codes))
print("validation cases:", len(validation_cases))
print("example query:", validation_query_texts[0])
print("target:", validation_cases[0]["target"], occupation_label(metadata_lookup, validation_cases[0]["target"]))

## Dense Retrieval Baseline

In [ ]:
model = SentenceTransformer(EMBEDDING_MODEL, device=device)

candidate_embeddings = model.encode(
    candidate_texts,
    batch_size=EMBED_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
query_embeddings = model.encode(
    validation_query_texts,
    batch_size=EMBED_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

scores = query_embeddings @ candidate_embeddings.T
top_idx = np.argsort(-scores, axis=1)[:, :RETRIEVE_K]
dense_ranked = [[candidate_codes[j] for j in row] for row in top_idx]
dense_ranked_scores = [[float(scores[i, j]) for j in row] for i, row in enumerate(top_idx)]

In [ ]:
def evaluate_ranked_lists(cases, ranked_lists, ks=(1, 5, 10)):
    rows = []
    n = len(cases)
    ranks = []
    for case, ranked in zip(cases, ranked_lists):
        target = case["target"]
        try:
            rank = ranked.index(target) + 1
        except ValueError:
            rank = None
        ranks.append(rank)
    result = {"n": n, "MRR": float(np.mean([1.0 / r if r else 0.0 for r in ranks]))}
    for k in ks:
        result[f"Recall@{k}"] = float(np.mean([1.0 if r and r <= k else 0.0 for r in ranks]))
    return result, ranks

baseline_metrics, baseline_ranks = evaluate_ranked_lists(validation_cases, dense_ranked, METRIC_KS)
baseline_metrics

## LLM Filtering

For each query, the LLM receives the dense top candidates and assigns each candidate a score:

- `0`: not plausible
- `1`: weakly plausible
- `2`: plausible
- `3`: highly plausible

Then the notebook filters candidates with `score >= FILTER_THRESHOLD` while preserving the original dense order.

In [ ]:
def compact_candidate_doc(code):
    meta = metadata_lookup.get(code, {})
    label = meta.get("label") or f"ESCO code {code}"
    description = meta.get("description") or ""
    if len(description) > 260:
        description = description[:260] + "..."
    return {"code": code, "title": label, "description": description}

def build_llm_prompt(query, candidates):
    candidate_payload = [compact_candidate_doc(code) for code in candidates]
    return f"""
You are evaluating next-occupation plausibility for ESCO occupations.

Given a person's career history query and a list of candidate ESCO occupations, assign each candidate a plausibility score as the person's next occupation.

Score scale:
0 = not plausible or unrelated
1 = weakly plausible
2 = plausible
3 = highly plausible

Return only a valid JSON object whose keys are ESCO codes and whose values are integers from 0 to 3. Do not include explanations.

Career history query:
{query}

Candidate ESCO occupations:
{json.dumps(candidate_payload, ensure_ascii=False)}
""".strip()

def parse_json_object(text):
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if match:
            return json.loads(match.group(0))
        raise

def score_candidates_with_openai_compatible(query, candidates, max_retries=3):
    from openai import OpenAI
    client_kwargs = {}
    if OPENAI_BASE_URL:
        client_kwargs["base_url"] = OPENAI_BASE_URL
    client = OpenAI(**client_kwargs)
    prompt = build_llm_prompt(query, candidates)
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[
                    {"role": "system", "content": "You are a strict JSON-only classifier for occupation plausibility."},
                    {"role": "user", "content": prompt},
                ],
                temperature=0,
            )
            content = response.choices[0].message.content
            raw = parse_json_object(content)
            return {code: int(raw.get(code, 0)) for code in candidates}
        except Exception as exc:
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)

def filter_ranked_list(ranked, llm_scores, threshold):
    return [code for code in ranked if int(llm_scores.get(code, 0)) >= threshold]

In [ ]:
llm_scores_per_case = []
filtered_ranked = []
failures = []

if RUN_LLM:
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("Set OPENAI_API_KEY before running LLM filtering, or set RUN_LLM = False.")
    for i, (query, ranked) in enumerate(tqdm(list(zip(validation_query_texts, dense_ranked)), desc="LLM filtering")):
        llm_candidates = ranked[:LLM_TOP_K]
        try:
            llm_scores = score_candidates_with_openai_compatible(query, llm_candidates)
        except Exception as exc:
            failures.append({"case_index": i, "error": repr(exc)})
            llm_scores = {code: 0 for code in llm_candidates}
        llm_scores_per_case.append(llm_scores)
        filtered = filter_ranked_list(ranked, llm_scores, FILTER_THRESHOLD)
        filtered_ranked.append(filtered)
else:
    print("RUN_LLM is False; skipping LLM filtering.")

## Compare Metrics

In [ ]:
if RUN_LLM:
    filtered_metrics, filtered_ranks = evaluate_ranked_lists(validation_cases, filtered_ranked, METRIC_KS)
    gold_retention = np.mean([
        1.0 if case["target"] in filtered else 0.0
        for case, filtered in zip(validation_cases, filtered_ranked)
    ])
    avg_before = np.mean([min(LLM_TOP_K, len(ranked)) for ranked in dense_ranked])
    avg_after = np.mean([len(filtered) for filtered in filtered_ranked])
    summary = pd.DataFrame([
        {"setting": "dense_baseline", **baseline_metrics, "gold_retention": 1.0, "avg_candidates": avg_before},
        {"setting": f"llm_filter_score_ge_{FILTER_THRESHOLD}", **filtered_metrics, "gold_retention": float(gold_retention), "avg_candidates": float(avg_after)},
    ])
    display(summary)
    print("LLM failures:", len(failures))
else:
    display(pd.DataFrame([{ "setting": "dense_baseline", **baseline_metrics }]))

In [ ]:
if RUN_LLM:
    rows = []
    for i, case in enumerate(validation_cases):
        target = case["target"]
        rows.append({
            "case_index": i,
            "person_id": case["person_id"],
            "target_code": target,
            "target_label": occupation_label(metadata_lookup, target),
            "baseline_rank": baseline_ranks[i],
            "filtered_rank": filtered_ranks[i],
            "target_retained": target in filtered_ranked[i],
            "num_filtered_candidates": len(filtered_ranked[i]),
            "query": validation_query_texts[i],
            "dense_top10": dense_ranked[i][:10],
            "filtered_top10": filtered_ranked[i][:10],
            "llm_scores": llm_scores_per_case[i],
        })
    details = pd.DataFrame(rows)
    display(details.head(10))
    details.to_csv("jobhop_llm_filtering_pilot_details.csv", index=False)
    summary.to_csv("jobhop_llm_filtering_pilot_summary.csv", index=False)
    with open("jobhop_llm_filtering_pilot_failures.json", "w", encoding="utf-8") as f:
        json.dump(failures, f, indent=2, ensure_ascii=False)
    print("Saved jobhop_llm_filtering_pilot_summary.csv")
    print("Saved jobhop_llm_filtering_pilot_details.csv")
    print("Saved jobhop_llm_filtering_pilot_failures.json")

## How to Interpret

Useful signs:

- `MRR` improves after filtering.
- `Recall@5` or `Recall@10` improves after filtering.
- `gold_retention` stays high.
- `avg_candidates` decreases meaningfully.

Warning signs:

- `gold_retention` is low: the LLM is filtering out too many correct targets.
- `MRR` drops strongly: the filter is hurting ranking quality.
- many LLM failures: prompt, API, or output parsing needs adjustment.

For a proper final experiment, choose the prompt, `LLM_TOP_K`, and `FILTER_THRESHOLD` on validation only, then freeze them and evaluate test once.